# Sesión 01 · Modelos en sistemas híbridos y guardarraíles

Construimos un **chatbot de soporte** para SentimentLab que une un **modelo HF** (detecta la intención) con un **motor de reglas** (decide), protegido por un **guardarraíl** de dominio.

Objetivos:
1. Usar `zero-shot-classification` para detectar intención.
2. Construir un mini motor de reglas encima del modelo.
3. Añadir un guardarraíl de confianza.

In [ ]:
%pip install -q transformers

## 1 · Detectar intención con zero-shot classification

In [ ]:
from transformers import pipeline

clasificador = pipeline("zero-shot-classification")

texto = "Quiero devolver un producto que llegó roto"
etiquetas = ["consulta", "reclamacion", "compra", "despedida"]

r = clasificador(texto, etiquetas)
print("intención:", r["labels"][0], "| confianza:", round(r["scores"][0], 3))

## 2 · El motor de reglas encima del modelo

In [ ]:
REGLAS = {
    "reclamacion": "Escalar a agente humano",
    "compra":       "Ayudar a finalizar la compra",
    "consulta":     "Responder con el FAQ",
    "despedida":    "Despedirse y cerrar la conversación",
}

def decidir(texto):
    r = clasificador(texto, list(REGLAS.keys()))
    intencion, score = r["labels"][0], r["scores"][0]
    if score < 0.5:                       # guardarraíl de confianza
        return intencion, "Preguntar qué necesita el cliente"
    return intencion, REGLAS[intencion]

for frase in ["Me llegó roto el paquete", "Quiero comprar otro", "¿Tenéis envío a Valencia?", "hola"]:
    intencion, accion = decidir(frase)
    print(f"'{frase}' -> [{intencion}] {accion}")

## 3 · Guardarraíl de toxicidad

Un segundo guardarraíl que bloquea mensajes ofensivos antes de responder.

In [ ]:
tox = pipeline("text-classification", model="unitary/toxic-bert")

def es_toxico(texto):
    r = tox(texto)[0]
    return r["label"] == "toxic" and r["score"] > 0.7

print(es_toxico("Me ha encantado el servicio"))
print(es_toxico("sois una mierda de empresa"))

## Práctica propuesta (miniproyecto)

1. Añade **2 intenciones** más a `REGLAS` (p. ej. `seguimiento`, `queja`).
2. Integra el guardarraíl de toxicidad: si `es_toxico(texto)`, responder "Mensaje no permitido".
3. Prueba con varias frases y comenta el resultado.

> Entrega: este notebook ejecutado.

In [ ]:
# TODO: amplía REGLAS con 2 intenciones y añade el guardarraíl de toxicidad a decidir()